In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
!pip install transformers datasets scikit-learn pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!unzip Fake.csv.zip
!unzip True.csv.zip

In [ ]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

In [ ]:
fake.head()

In [ ]:
true.head()

In [ ]:
fake["label"] = 0
true["label"] = 1

In [ ]:
data = pd.concat([fake, true])

In [ ]:
data = data.sample(frac=1).reset_index(drop=True)

In [ ]:
data.head()

In [ ]:
data["label"].value_counts()

In [ ]:
data["content"] = data["title"] + " " + data["text"]

In [ ]:
data = data[["content", "label"]]

In [ ]:
data.head()

In [ ]:
print(data["content"][0])

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    data["content"].tolist(),
    data["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
print(len(train_texts))
print(len(val_texts))

In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
tokenizer(train_texts[0])

In [ ]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=512
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=512
)

In [ ]:
print(train_encodings.keys())

In [ ]:
list(train_encodings.keys())

In [ ]:
len(train_encodings["input_ids"])

In [ ]:
class FakeNewsDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = FakeNewsDataset(train_encodings, train_labels)
val_dataset = FakeNewsDataset(val_encodings, val_labels)

In [ ]:
len(train_dataset)

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    logging_steps=100,
    save_steps=500
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
 trainer.train()

In [ ]:
trainer.save_model("fake_news_distilbert")
tokenizer.save_pretrained("fake_news_distilbert")

In [ ]:
!zip -r fake_news_distilbert.zip fake_news_distilbert

In [ ]:
from google.colab import files
files.download("fake_news_distilbert.zip")

In [ ]:
import numpy as np

predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(val_labels, preds)
print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(val_labels, preds))

In [ ]:
cm = confusion_matrix(val_labels, preds)

sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
wrong = []

for i in range(len(preds)):
    if preds[i] != val_labels[i]:
        wrong.append((val_texts[i], preds[i], val_labels[i]))

len(wrong)

In [ ]:
import torch

def predict_news(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)

    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    prediction = torch.argmax(logits, dim=1).item()

    if prediction == 0:
        return "Fake News"
    else:
        return "Real News"

In [ ]:
predict_news("Breaking: Government announces new economic policy to improve employment rates")

In [ ]:
predict_news("Shocking! Scientists confirm aliens secretly control the world government")

In [ ]:
import torch.nn.functional as F

def predict_news_with_confidence(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = F.softmax(logits, dim=1)

    confidence, prediction = torch.max(probs, dim=1)

    label = "Fake News" if prediction.item() == 0 else "Real News"

    return label, confidence.item()

In [ ]:
predict_news_with_confidence("Breaking: Government announces new tax reforms for businesses")

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def fake_news_detector(text):
    label, confidence = predict_news_with_confidence(text)
    return f"{label} (Confidence: {confidence*100:.2f}%)"

interface = gr.Interface(
    fn=fake_news_detector,
    inputs=gr.Textbox(lines=5, placeholder="Enter news text here..."),
    outputs="text",
    title="Fake News Detection using DistilBERT",
    description="Enter a news headline or article to check if it is Fake or Real."
)

interface.launch()